# GRU live-sequence — next goal în 10' (Colab GPU)
Antrenează modelul **v1 fără momentum** pe `ml/live_seq_full` (train 2024+2025 / test 2026),
cu temperature scaling pe ultima felie temporală din train, apoi evaluează vs baseline-uri.

Codul modelului = **sursa unică** `ml/experiment_live_sequence.py` (același validat pe VPS) —
notebook-ul doar îl rulează (device cuda automat). NU atinge motorul de producție.

## 1) GPU + dependențe
torch & numpy sunt preinstalate pe Colab. Verificăm GPU-ul:

In [ ]:
import torch
print('CUDA disponibil:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('torch', torch.__version__)

## 2) Adu bundle-ul (dataset + script) de pe VPS
**Metoda A (link public, fără auth):** pune URL-ul dat de `transfer.sh` la pasul de pe VPS.
Bundle-ul conține `ml/live_seq_full/` + `ml/experiment_live_sequence.py`.

In [ ]:
URL = 'PUNE_AICI_URL_DE_LA_TRANSFER_SH'  # ex: https://transfer.sh/xxxx/live_seq_bundle.tgz
!wget -q -O bundle.tgz "$URL" && tar xzf bundle.tgz && echo OK && ls -la ml/ && du -sh ml/live_seq_full

**Metoda B (Google Drive):** urcă `live_seq_bundle.tgz` în Drive de pe iPhone, apoi:
```python
from google.colab import drive; drive.mount('/content/drive')
!tar xzf "/content/drive/MyDrive/live_seq_bundle.tgz" && ls ml/
```

## 3) ANTRENARE (GRU + embeddings gazdă/oaspete/ligă) + temperature scaling + checkpoint
`--train` rulează pe cuda automat, salvează model+temperatură+encoders+scaler în `ml/live_seq.pt`,
și face și eval la final. Split: train 2024+2025 / test 2026 / val = ultima felie din 2025.

In [ ]:
!python ml/experiment_live_sequence.py --train --data ml/live_seq_full --ckpt ml/live_seq.pt --report ml/live_seq_eval.txt

## 4) EVAL pe test 2026 — model vs baseline-uri corecte
(a) base-rate marginal din train (prag minim) · (b) Poisson no-xG (ținta reală).
Brier multiclass + binar (colaps any-goal) + ECE + reliability.

In [ ]:
!python ml/experiment_live_sequence.py --eval --data ml/live_seq_full --ckpt ml/live_seq.pt --report ml/live_seq_eval.txt
print('\n--- raport ---'); print(open('ml/live_seq_eval.txt').read())

## 5) Descarcă checkpoint-ul (înapoi pe iPhone / Drive)

In [ ]:
from google.colab import files
files.download('ml/live_seq.pt')
files.download('ml/live_seq_eval.txt')

> (c) head-to-head vs heuristica reală `calcNextGoalWindow(10)` se rulează **pe VPS**
> (are nevoie de `live_stats` din DB): `--eval-livesubset` cu același `--ckpt` și `--data`.